In [ ]:

!pip uninstall torch -y
!pip uninstall torchvision -y


!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torchvision --index-url https://download.pytorch.org/whl/cu118

!pip install transformers accelerate
!pip install datasets

!pip show transformers
!pip show datasets
!pip show torch
!pip show accelerate
!pip show torchvision
!pip show torchaudio


In [ ]:
from transformers import Trainer, TrainingArguments
import pandas as pd
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# connect wandb for visualization and tracking

# !pip install wandb -q

# import wandb
# wandb.login()
# wandb.init(project="PhobertDescribetion2Expert",name="Deex_o1")


In [ ]:


data = []
# Thay đổi đường dẫn này cho phù hợp với vị trí tệp của bạn trên Google Drive
file_path = '/content/drive/My Drive/crawl_data/data.json'
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
print(df.head())

In [ ]:
labels = [
        "Software Engineer", "Backend Developer", "Fullstack Developer", "Mobile Developer",
        "Frontend Developer", "Blockchain Engineer", "Kinh doanh phần mềm", "Sales IT Phần mềm khác",
        "Kinh doanh Domain/Hosting/Server", "Thiết kế đồ họa (Graphic Design)", "Animation Design",
        "3D Molder",  "Illustration", "IT Helpdesk/IT support",
        "DevOps Engineer", "Kỹ thuật IT", "System Administrator", "Network Engineer", "System Engineer",
        "Database Administrator (DBA)", "Cloud Engineer", "Software Tester (Automation & Manual)",
        "Manual Tester", "Automation Tester", "QA Engineer", "Process Quality Assurance (PQA)",
        "Game Tester", "Business Analyst (Phân tích nghiệp vụ)", "Product Owner/Product Manager",
        "Product Analyst/Research", "Thiết kế đồ họa (Graphic Design)", "UI/UX Design",
        "3D Molder",  "IT Consultant",
        "Bán hàng kỹ thuật IT", "BIM Engineer", "IT Project Manager", "Kỹ sư cầu nối BrSE",
        "IT Comtor", "Scrum Master", "Data Analyst", "Data Engineer", "Data Scientist"
    ]
print("so luong label: ",len(labels))



In [ ]:
# Loại bỏ các dòng có nhãn không hợp lệ
df = df[df['chuyen_mon'].isin(labels)]

# Tạo ánh xạ nhãn sang ID
label_map = {label: i for i, label in enumerate(labels)}
df['label'] = df['chuyen_mon'].map(label_map)

# Chuẩn bị dữ liệu cho huấn luyện
texts = df['mo_ta_cong_viec'].tolist()
label_ids = df['label'].tolist()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Lọc dữ liệu theo danh sách nhãn đã định nghĩa
df = df[df['chuyen_mon'].isin(labels)]

# 1. Kiểm tra phân bố nhãn
label_counts = df['chuyen_mon'].value_counts()
print("Số lượng mẫu cho mỗi nhãn:")
print(label_counts)

# 2. Tính toán tỷ lệ
label_percentages = df['chuyen_mon'].value_counts(normalize=True) * 100
print("\nTỷ lệ phần trăm mẫu cho mỗi nhãn:")
print(label_percentages)

# 3. Trực quan hóa phân bố nhãn
plt.figure(figsize=(15, 8))
sns.countplot(y='chuyen_mon', data=df, order=label_counts.index, palette='viridis')
plt.title('Phân bố số lượng mẫu theo Chuyên môn (Nhãn)')
plt.xlabel('Số lượng mẫu')
plt.ylabel('Chuyên môn (Nhãn)')
plt.tight_layout()
plt.show()

In [ ]:


train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, label_ids, test_size=0.2, random_state=42, stratify=label_ids
)
#check train_texts 100 record
for i in range(100):
    print("description :",train_texts[i])
    print("expert :",labels[train_labels[i]])

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=len(labels))
# Kiểm tra và chuyển mô hình sang GPU nếu có
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)

In [ ]:


class JobDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)
train_dataset = JobDataset(train_encodings, train_labels)
val_dataset = JobDataset(val_encodings, val_labels)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1=f1_score(labels, preds, average='weighted')
    acc=accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

In [ ]:
# when start train model, please leave comment

# training_args = TrainingArguments(
#     output_dir='./content/drive/My Drive/phobert_job_classification_model',
#     num_train_epochs=100,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     warmup_steps=500,
#     weight_decay=0.01,
#     logging_dir='./logs',
#     logging_steps=10,
#     # evaluation_strategy="epoch",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     report_to="wandb"
# )
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )
# trainer.train()

In [ ]:
# close wandb

# results = trainer.evaluate(val_dataset)
# print(results)

# wandb.finish()

In [ ]:
# test

output_dir = '/content/drive/My Drive/phobert_job_classification_model/checkpoint-5504'
model = AutoModelForSequenceClassification.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


test_texts = [
    '''Giới thiệu các giải pháp về phần mềm tích hợp như API Management, MQ, Event Streaming, ESB… với khách hàng chủ yếu trong lĩnh vực Tài chính, Ngân hàng.
Phối hợp cùng các bạn kỹ sư Amigo, hãng, đối tác thực hiện demo, PoC tính năng sản phẩm, làm thầu, triển khai các dự án về giải pháp phần mềm tích hợp.
Nghiên cứu các giải pháp, công nghệ theo định hướng của công ty.
Các công việc khác theo yêu cầu''',
    '''- Lập trình và phát triển app di động bằng ngôn ngữ Kotlin ưu tiên biết thêm React Native, đã từng làm Google Admob.
Tham gia xây dựng, nâng cấp và sửa lỗi cho các ứng dụng đã có.
Phối hợp với team thiết kế, backend và tester để hoàn thiện sản phẩm.''',
    '''Lên ý tưởng, thiết kế các hạng mục sản phẩm quảng cáo theo yêu cầu: catalogue, thiết kế sân khấu, backdrop, banner,...
Hỗ trợ các bộ phận khác các vấn đề liên quan đến chuyên môn.
Đảm bảo chất lượng hình ảnh và thời gian hoàn thành đúng deadline.
Cập nhật xu hướng thiết kế mới, ứng dụng vào công việc.'''

]


test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=256, return_tensors="pt")


test_encodings = {key: tensor.to(device) for key, tensor in test_encodings.items()}


with torch.no_grad():
    outputs = model(**test_encodings)


probabilities = torch.softmax(outputs.logits, dim=1)


predicted_label_indices = torch.argmax(probabilities, dim=1)


id_to_label = {i: label for label, i in label_map.items()}
predicted_labels = [id_to_label[idx.item()] for idx in predicted_label_indices]


for text, label in zip(test_texts, predicted_labels):
    print(f"desribetion: '{text}'")
    print(f"Predicted label: {label}\n")